# SoccerNet Dense Video Captioning: chạy trên Colab (v3.2)

Xem `CHANGELOG.md` để biết thay đổi của từng phiên bản.

Notebook này train và test dự án **soccernet-caption-deltalab** (QFormer + GPT-2) trên một phần nhỏ của dataset SoccerNet `caption-2024`.

**Chuẩn bị trước khi chạy:**
1. Chọn GPU: `Runtime → Change runtime type → A100 GPU` hoặc `T4 GPU`.
2. Chạy lần lượt từng cell từ trên xuống. Cell 4 tự `git clone` code từ GitHub.

**Khi code trên GitHub có bản mới:** chỉ cần chạy lại **cell 4** (`git pull`), rồi chạy lại cell 8. Dữ liệu đã tải không bị mất.

Dữ liệu (feature và nhãn) là công khai: notebook tải thẳng từ HuggingFace về ổ của Colab, không cần NDA hay token. Model đã train được lưu vào Drive, nên vẫn còn sau khi hết phiên Colab.

**Chạy trên tài khoản Google khác:** chủ Drive chia sẻ thư mục `soccer_project` cho tài khoản đó. Người chạy bấm chuột phải vào thư mục → **Sắp xếp → Thêm lối tắt vào Drive**, đặt lối tắt ở `Drive của tôi`. Nếu đặt ở chỗ khác hoặc đổi tên, sửa `DRIVE_ROOT` ở cell 2; cell 3 cũng tự tìm thư mục `soccer_project`.

**Mới ở v3.0, mục 12 (cách B):** chạy model đã train trên feature Baidu với **video** thay cho file npy có sẵn:
video → TimeSformer của Baidu (768 chiều) → bộ chuyển đổi 768 → 8576 → model cũ, không train lại.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình
Sửa các giá trị ở đây. Đặt `-1` để lấy toàn bộ split: train 281, valid 92, test 98 trận, khoảng 89 GB.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"  # repo code
GIT_BRANCH = "main"
# Thư mục dự án trên Drive của NGƯỜI ĐANG CHẠY (bản gốc, hoặc lối tắt tới thư mục được chia sẻ)
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"   # nơi lưu / đọc model đã train

N_TRAIN, N_VALID, N_TEST = 20, 5, 5  # số trận mỗi split (~190 MB/trận), 30 trận ~5,7 GB

MODEL_NAME = "gpt2-30games" # tên thư mục kết quả trong models/
GPT_TYPE = "gpt2"           # "gpt2" (124M) hoặc "gpt2-medium" (355M)
MAX_EPOCHS = 10             # số epoch train caption; README dùng mặc định 15
EVAL_FREQ = 5               # chấm METEOR/BLEU trên valid mỗi N epoch (sinh câu chậm, đừng để 1)
BATCH_SIZE = 32
NUM_WORKERS = 2             # Colab chỉ có ít CPU, để 4 sẽ bị cảnh báo chậm
NMS_THRESHOLD = 0.7

## 3. Mount Google Drive

Khi cửa sổ xin quyền hiện ra, chọn đúng tài khoản Google rồi bấm **Allow / Cho phép** cho mọi quyền được hỏi.
Nếu vẫn báo `mount failed`, bạn có thể bỏ qua: model sẽ được lưu tạm trên Colab, và cell 9 có cách tải về máy.

In [ ]:
from google.colab import drive
import os, glob
try:
    drive.mount("/content/drive", force_remount=True)
    USE_DRIVE = True
except Exception as e:
    USE_DRIVE = False
    print("Không mount được Drive:", e)
    print("Tiếp tục chạy, model sẽ lưu tạm trong /content (mất khi hết phiên Colab).")

if USE_DRIVE and not os.path.isdir(DRIVE_ROOT):
    # tìm thư mục soccer_project ở chỗ khác trong Drive (lối tắt, thư mục con, Shared drives)
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    if found:
        DRIVE_ROOT = found[0]
        DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"
    else:
        print(f"Không thấy {DRIVE_ROOT}. Hãy thêm lối tắt tới thư mục soccer_project được chia sẻ "
              "vào 'Drive của tôi', hoặc sửa DRIVE_ROOT ở cell 2.")
if USE_DRIVE:
    print("DRIVE_ROOT =", DRIVE_ROOT)
    !ls "{DRIVE_ROOT}"


## 4. Lấy code từ GitHub
Lần đầu thì `git clone`. Các lần sau thì `git pull` để cập nhật code mới nhất, còn `dataset/` và `models/` vẫn giữ nguyên.

In [ ]:
import os, glob
WORK_DIR = "/content/soccernet-caption-deltalab"
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -3
!grep -m1 "^## v" CHANGELOG.md

## 5. Cài thư viện

- `nlg-eval` được cài với `--no-deps` vì các thư viện phụ thuộc cũ của nó (gensim 3.8, Theano) không cài được trên Python mới. Code chỉ dùng BLEU, METEOR, ROUGE và CIDEr nên không cần đến chúng.
- METEOR cần Java, nên cell này cài Java nếu Colab chưa có.
- Java trên Colab in thêm các dòng cảnh báo `Cgroup ...` vào stdout, làm METEOR đọc nhầm điểm (`ValueError: could not convert string to float`). `JAVA_TOOL_OPTIONS` tắt các cảnh báo này.

In [ ]:
!pip install -q SoccerNet tiktoken transformers wandb pycocoevalcap psutil six
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master
!which java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null)

import os
os.environ["JAVA_TOOL_OPTIONS"] = "-XX:-UseContainerSupport -Xlog:disable"
!java -version

Kiểm tra nhanh METEOR. Nếu cell này in ra một con số thì bước chấm điểm sẽ chạy được.

In [ ]:
from nlgeval.pycocoevalcap.meteor.meteor import Meteor
m = Meteor()
print("METEOR test:", m.compute_score({0: ["a player takes a free kick"]}, {0: ["a player is awarded a free kick"]})[0])
del m

## 6. Xem danh sách trận rồi tải dữ liệu

Các trận được chia theo **danh sách chính thức của SoccerNet**: mỗi trận chỉ thuộc một split, nên train, valid và test không trùng nhau.
Trong mỗi split, script lấy `N` trận đầu danh sách. Danh sách xếp theo giải rồi theo ngày.

**6a.** Xem trước danh sách trận sẽ dùng. Cell này không tải gì. Muốn đổi số trận thì sửa cell 2 rồi chạy lại cell này.

In [ ]:
!python download_subset.py dataset/caption-2024 --train {N_TRAIN} --valid {N_VALID} --test {N_TEST} --list_only

**6b.** Tải feature và nhãn của các trận trên từ HuggingFace, khoảng 190 MB mỗi trận.

In [ ]:
!python download_subset.py dataset/caption-2024 --train {N_TRAIN} --valid {N_VALID} --test {N_TEST}
!du -sh dataset

## 7. Lưu `models/` thẳng lên Drive
Tạo symlink `models → Drive`, để checkpoint và kết quả không bị mất khi hết phiên Colab. Nếu chưa mount được Drive thì `models/` nằm trên ổ Colab.

In [ ]:
if USE_DRIVE:
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    if not os.path.islink("models"):
        !rm -rf models
    !ln -sfn "{DRIVE_MODEL_DIR}" models
else:
    os.makedirs("models", exist_ok=True)
!ls -la models

## 8. Train và test

`main.py` lần lượt chạy:
1. **classifying**: 10 epoch, phân loại sự kiện.
2. **captioning**: `MAX_EPOCHS` epoch, sinh lời bình bằng GPT-2.
3. **spotting**: 15 epoch, tìm thời điểm xảy ra sự kiện.
4. **dvc**: ghép bước 2 và 3, rồi chấm METEOR/BLEU/CIDEr trên test.

Hai biến môi trường được bật:
- `SN_LOCAL_SUBSET=1`: chỉ dùng các trận đã tải.
- `WANDB_MODE=offline`: không cần tài khoản wandb.

In [ ]:
os.environ["SN_LOCAL_SUBSET"] = "1"
os.environ["WANDB_MODE"] = "offline"
os.environ["JAVA_TOOL_OPTIONS"] = "-XX:-UseContainerSupport -Xlog:disable"  # để METEOR đọc đúng điểm

!python main.py     --SoccerNet_path dataset/caption-2024     --model_name {MODEL_NAME}     --GPU 0     --pool QFormer     --model_type gpt     --gpt_type {GPT_TYPE}     --NMS_threshold {NMS_THRESHOLD}     --split_test test     --max_epochs {MAX_EPOCHS}     --evaluation_frequency {EVAL_FREQ}     --batch_size {BATCH_SIZE} \
    --max_num_worker {NUM_WORKERS}

## 9. Xem kết quả

In [ ]:
import glob
log = sorted(glob.glob(f"models/{MODEL_NAME}/*.log"))[-1]
print("Log:", log)
!grep -E "METEOR|Bleu_4|CIDEr|a_mAP" "{log}" | tail -30

In [ ]:
# Xem vài câu bình luận model sinh ra cho trận test đầu tiên
import json
files = sorted(glob.glob(f"models/{MODEL_NAME}/outputs/test/**/results_dense_captioning.json", recursive=True))      or sorted(glob.glob(f"models/{MODEL_NAME}/outputs/**/*.json", recursive=True))
print(files[:5])
if files:
    preds = json.load(open(files[0]))
    for p in preds.get("predictions", preds.get("annotations", []))[:15]:
        print(p.get("gameTime"), "|", p.get("label"), "|", p.get("comment", p.get("anonymized")))

## 10. Chỉnh lại ngưỡng NMS và cách sinh câu (không train lại)

Dùng model spotting và caption **đã train** trong `models/{MODEL_NAME}`:
- **NMS_threshold** thấp hơn thì báo nhiều sự kiện hơn và ít bỏ sót hơn (recall cao), nhưng dễ báo sai hơn.
- **top_k** = 1 luôn chọn từ chắc nhất nên dễ lặp câu; top_k > 1 chọn ngẫu nhiên trong k từ tốt nhất nên câu đa dạng hơn.

**10a.** Thử các tổ hợp trên **valid**, rồi chọn tổ hợp có `METEOR_loose` cao nhất. Bảng kết quả lưu ở `models/{MODEL_NAME}/retune_valid.csv`.

In [ ]:
NMS_LIST = "0.3 0.4 0.5 0.7"
TOPK_LIST = "1 3"
!python retune.py --SoccerNet_path dataset/caption-2024 --model_name {MODEL_NAME} --split valid --nms_thresholds {NMS_LIST} --top_k {TOPK_LIST}

**10b.** Áp dụng tổ hợp tốt nhất lên **test**. Điền 2 giá trị dòng `best on valid` in ở cell trên. Cell này ghi lại `outputs/test`, timeline ở bước 11 sẽ dùng kết quả mới.

In [ ]:
BEST_NMS, BEST_TOPK = 0.4, 3   # sửa theo dòng "best on valid"
!python retune.py --SoccerNet_path dataset/caption-2024 --model_name {MODEL_NAME} --split test --nms_thresholds {BEST_NMS} --top_k {BEST_TOPK}

## 11. Timeline bình luận: luồng YES + luồng NO

- **Luồng YES (có sự kiện):** lời bình do model sinh ra tại các thời điểm spotting tìm được (`results_dense_captioning.json`).
- **Luồng NO (không có sự kiện):** ở những khoảng im lặng từ 45 giây trở lên, chèn bình luận ngoài lề lấy từ:
  - metadata SoccerNet: sân, vòng đấu, trọng tài, khán giả, HLV, sơ đồ, đội hình, đội trưởng, dự bị, quốc tịch cầu thủ;
  - diễn biến **đã xảy ra tới thời điểm đó**: tỉ số, thẻ phạt, thay người (không dùng tỉ số chung cuộc);
  - dữ liệu crawl từ football-data.co.uk: bảng xếp hạng, phong độ 5 trận, lịch sử đối đầu, tính đến **trước ngày thi đấu** (không có cho Champions League).

Kết quả: `commentary_timeline.json` và `commentary_timeline.txt` trong thư mục của từng trận. Tin mới (bàn thắng, thẻ, thay người) được nói ngay khi biết, trễ tối đa khoảng 1 phút vì metadata chỉ ghi theo phút. Chuyện ngoài lề nhỏ chỉ được nói khi khoảng im lặng từ 2 phút trở lên, và không nói trong 2 phút sau bàn thắng. Tham số chỉnh được: `--spacing`, `--min_gap`, `--long_gap`, `--goal_cooldown`, `--after_event`, `--before_event`, `--no_crawl`.

In [ ]:
!python commentary_timeline.py --soccernet_path dataset/caption-2024 --predictions models/{MODEL_NAME}/outputs/test

In [ ]:
# Xem timeline của trận test đầu tiên: [EVENT] = luồng YES, [INFO ] = luồng NO
tl = sorted(glob.glob(f"models/{MODEL_NAME}/outputs/test/**/commentary_timeline.txt", recursive=True))
print(tl[0])
!head -60 "{tl[0]}"

## 12. Video → npy TimeSformer → ép thành 8576 → so độ giống với npy Baidu

**Chỉ cần chạy cell 1, 2, 3, 4, rồi 12a → 12d.** Không cần train, không cần cell 5–11.

1. **12a.** Cài thư viện và cấu hình.
2. **12b.** Giải nén video 224p từ Drive, tải npy Baidu gốc (và nhãn) của đúng các trận đó từ HuggingFace.
3. **12c.** Video → npy **768 chiều** bằng TimeSformer mà Baidu đã **fine-tune trên SoccerNet**. Code là bản PyTorch của model PaddlePaddle gốc, đã kiểm tra cho cùng kết quả (sai lệch < 1e-5). Mỗi giây một dòng, cùng số dòng với npy Baidu.
4. **12d.** **Ép 768 → 8576**: học phép chuyển đổi (ridge regression) trên các trận **train**, chọn tham số trên **valid**, rồi **đo độ giống với npy Baidu gốc trên các trận test**. Các trận test không được dùng để học.

Phép ép phải được học. Nếu chỉ thêm một lớp 768 → 8576 ngẫu nhiên thì các con số vô nghĩa, độ giống sẽ chỉ ở mức hai giây ngẫu nhiên (khoảng 0,67).

*12e–12f (tùy chọn, để sau):* đưa npy đã ép vào model caption cũ để chấm METEOR. Phần này cần model trong `sn-models` và cell 5, 7.

**Cần có trên Drive:** `soccer_project/dataset/Dataset.zip` (video) và file weights `.pdparams` của TimeSformer. Tải file weights bằng trình duyệt từ link OneDrive trong README của [baidu-research/Soccernet-features](https://github.com/baidu-research/Soccernet-features), rồi upload vào `soccer_project/weights/`.


**12a.** Cấu hình cho cách B.

In [ ]:
!pip install -q SoccerNet av   # av: đọc video
VIDEO_ZIP = f"{DRIVE_ROOT}/dataset/Dataset.zip"
VIDEO_DIR = "/content/videos_raw"
TS_DRIVE_DIR = f"{DRIVE_ROOT}/timesformer768"  # lưu feature đã trích, khỏi trích lại
MAPPER = f"{DRIVE_MODEL_DIR}/mapper_timesformer.pt" if USE_DRIVE else "models/mapper_timesformer.pt"  # + _report.json
TS_BATCH = 32          # T4: 32 (fp16). Nếu báo hết VRAM thì giảm xuống 16

import glob
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
print("weights:", weights)
TS_WEIGHTS = weights[0] if weights else None
assert os.path.exists(VIDEO_ZIP), f"Không thấy {VIDEO_ZIP}"
assert TS_WEIGHTS, "Chưa có file .pdparams trên Drive (xem hướng dẫn ở trên)"

**12b.** Giải nén video và tải nhãn + npy Baidu cho đúng các trận có video. Mất vài phút, chỉ cần chạy 1 lần mỗi phiên Colab.

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download
DATA = "dataset/caption-2024"

if not glob.glob(f"{VIDEO_DIR}/Dataset/**/*_224p.mkv", recursive=True):
    with zipfile.ZipFile(VIDEO_ZIP) as z:
        z.extractall(VIDEO_DIR, [m for m in z.namelist() if m.startswith("Dataset/") and m.endswith(".mkv")])

videos = sorted(glob.glob(f"{VIDEO_DIR}/Dataset/**/*_224p.mkv", recursive=True))
VIDEO_GAMES = sorted({os.path.relpath(os.path.dirname(v), f"{VIDEO_DIR}/Dataset") for v in videos})
print(len(videos), "video,", len(VIDEO_GAMES), "trận")

for g in VIDEO_GAMES:
    for f, repo, rev in [("Labels-caption.json", "SoccerNet/SN-Labels", "main"),
                         ("1_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings"),
                         ("2_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings")]:
        if not os.path.exists(f"{DATA}/{g}/{f}"):
            hf_hub_download(repo, f"{g}/{f}", repo_type="dataset", revision=rev, local_dir=DATA)

from SoccerNet.utils import getListGames
split_of = {g: s for s in ["train", "valid", "test", "challenge"] for g in getListGames([s], task="caption")}
for g in VIDEO_GAMES:
    print(f"{split_of.get(g, '?'):6} {g}")

**12c.** Trích feature TimeSformer, ghi `1_timesformer768.npy` và `2_timesformer768.npy` vào thư mục từng trận. Ước tính trên T4 khoảng 5–10 phút mỗi hiệp. Feature được sao lưu lên Drive (`TS_DRIVE_DIR`): lần sau cell này chép lại từ Drive và bỏ qua các hiệp đã có.

In [ ]:
import shutil
def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if USE_DRIVE and os.path.isdir(TS_DRIVE_DIR):
    sync(TS_DRIVE_DIR, DATA)   # đã trích ở phiên trước
!python timesformer_extract.py --weights "{TS_WEIGHTS}" --video_root {VIDEO_DIR}/Dataset --out_root {DATA} --batch_size {TS_BATCH}
if USE_DRIVE:
    sync(DATA, TS_DRIVE_DIR)

**12d.** Học bộ chuyển đổi 768 → 8576.
- Tự tìm độ lệch thời gian giữa hai loại feature.
- Ridge regression dùng 5 giây TimeSformer quanh giây t (`--context 2`).
- Chọn tham số trên các trận valid. Thêm `--mlp` để thử thêm một mạng MLP nhỏ, mạng này chỉ được giữ nếu tốt hơn ridge.

Cách đọc kết quả:

| Chỉ số | Ý nghĩa |
|---|---|
| `cosine` | Độ giống trung bình với npy Baidu thật, 1 = giống hệt |
| `cosine_ge_0.95` | Tỉ lệ số giây đạt cosine ≥ 0,95 |
| `cosine_TPN`… | Độ giống theo từng khối của 5 mạng |
| `baidu_next_second` | Mốc so sánh: Baidu giây t so với giây t+1 của chính nó (khoảng 0,94) |
| `baidu_random_second` | Mốc so sánh: hai giây ngẫu nhiên không liên quan (khoảng 0,67) |

In [ ]:
!python feature_mapper.py train --soccernet_path {DATA} --out {MAPPER} --train_splits train --val_splits valid --test_splits test

**12e. (tùy chọn, chấm caption)** Tạo npy giả Baidu (`baidu_from_timesformer.npy`) cho các trận test. Đồng thời tạo `baidu_ref.npy`: bản sao npy thật của **đúng các trận đó**, để so sánh công bằng.

In [ ]:
!python feature_mapper.py apply --soccernet_path {DATA} --mapper {MAPPER} --splits test --out_name baidu_from_timesformer.npy --ref_name baidu_ref.npy

**12f. (tùy chọn, cần cell 5 và 7)** Chạy **model cũ** (`{MODEL_NAME}`, không train lại) trên các trận test có video, lần lượt với npy thật và npy giả, rồi so điểm.
`SN_FEATURES` cho bộ chấm điểm biết dùng file feature nào để liệt kê các trận. Cell này ghi đè `outputs/test`: chạy lại 10b nếu cần kết quả test cũ.

In [ ]:
import pandas as pd
BEST_NMS, BEST_TOPK = 0.4, 3   # như 10b
rows = []
for name in ["baidu_ref.npy", "baidu_from_timesformer.npy"]:
    os.environ["SN_FEATURES"] = name
    !python retune.py --SoccerNet_path {DATA} --model_name {MODEL_NAME} --split test --features {name} --nms_thresholds {BEST_NMS} --top_k {BEST_TOPK}
    df = pd.read_csv(f"models/{MODEL_NAME}/retune_test.csv")
    df.insert(0, "features", name)
    df.to_csv(f"models/{MODEL_NAME}/retune_test_{name[:-4]}.csv", index=False)
    rows.append(df)
os.environ["SN_FEATURES"] = "baidu_soccer_embeddings.npy"
pd.concat(rows).T

Nếu không mount được Drive: nén kết quả (log, output, checkpoint caption) và tải về máy trước khi hết phiên Colab.

In [ ]:
if not USE_DRIVE:
    !cd models && zip -qr /content/{MODEL_NAME}_results.zip {MODEL_NAME} -x "*/classifying/*" "*/spotting/*"
    from google.colab import files as colab_files
    colab_files.download(f"/content/{MODEL_NAME}_results.zip")
else:
    print("Kết quả đã nằm trên Drive:", DRIVE_MODEL_DIR)